# 05 Test/Predict
Use Ultralytics API directly (`model.predict`)


In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil

REPO_URL = 'https://github.com/ahmedelbamby-aast/yolov13-custom.git'
WORKING_REPO = Path('/kaggle/working/yolov13-custom')
DEV_REPO = Path('/kaggle/work_here/yolov13')

if not WORKING_REPO.exists():
    subprocess.check_call(['git', 'clone', REPO_URL, str(WORKING_REPO)])
if not DEV_REPO.exists():
    DEV_REPO.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(WORKING_REPO, DEV_REPO, dirs_exist_ok=True)

os.chdir(DEV_REPO)
if str(DEV_REPO) not in sys.path:
    sys.path.insert(0, str(DEV_REPO))

os.environ['LD_LIBRARY_PATH'] = '/usr/local/nvidia/lib64:/usr/local/cuda/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')
os.environ.setdefault('Y13_USE_TURING_FLASH', '1')
print('repo:', DEV_REPO)


In [ ]:
import subprocess, sys
pkgs = ['-U','pip','setuptools','wheel']
subprocess.check_call([sys.executable,'-m','pip','install',*pkgs])
subprocess.check_call([sys.executable,'-m','pip','install','-e','.','rich'])
print('env ready')


In [ ]:
import os, torch
print('torch', torch.__version__, 'cuda', torch.version.cuda)
print('cuda_available', torch.cuda.is_available(), 'count', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))
from ultralytics.nn.modules import block
print('flash backend:', getattr(block,'FLASH_BACKEND','unknown'))
print('flash enabled:', getattr(block,'USE_FLASH_ATTN', False))
print('Y13_USE_TURING_FLASH=', os.getenv('Y13_USE_TURING_FLASH'))


In [ ]:
from ultralytics import YOLO
model = YOLO('/kaggle/working/y13_runs/notebook_train/weights/best.pt')
out = model.predict(
    source='/kaggle/work_here/datasets/coco128/images/train2017',
    imgsz=640,
    conf=0.25,
    iou=0.7,
    device=0,
    save=True,
    save_txt=True,
    save_conf=True,
    visualize=True,
    project='/kaggle/working/y13_runs/notebook_test',
    name='predict_run',
    exist_ok=True,
)
print('predictions', len(out))


In [ ]:
from ultralytics import YOLO
model = YOLO('/kaggle/working/y13_runs/notebook_train/weights/best.pt')
out = model.predict(
    source='/kaggle/work_here/datasets/coco128/images/train2017',
    imgsz=640,
    conf=0.25,
    iou=0.7,
    device=0,
    augment=True,
    save=True,
    project='/kaggle/working/y13_runs/notebook_test',
    name='predict_tta',
    exist_ok=True,
)
print('tta predictions', len(out))
